# 📈 Data Designer Tutorial: Investment Advisor Conversations

#### 📚 What you'll learn

This notebook demonstrates the basics of Data Designer by generating an "Investment Advisor Conversation" dataset.

Example scenario:
- Customer: "Should I buy NVIDIA?"
- Advisor: "Let's discuss your risk tolerance."

Data Designer can generate:
- Different client profiles (Risk Preference, Investment Goal, Time Horizon, Asset Allocation, Portfolio)
- Four types of advisor recommendations:
  - **Suitable Recommendation** – aligns with client's risk profile and goals
  - **Unsuitable Recommendation** – conflicts with client's risk profile
  - **Hallucinated Recommendation** – contains invented facts, fake products, or made-up data
  - **Compliance Violation** – breaks regulations (guaranteed returns, insider info, etc.)

This dataset is highly valued by securities firms for compliance training and AI safety.

#### 📚 学習内容

このノートブックでは、「投資アドバイザーとの会話」データセットを生成することで、Data Designerの基本操作を解説します。

例：
- 顧客：「NVIDIA株を買うべきでしょうか？」

- アドバイザー：「お客様のリスク許容度についてお話ししましょう。」

Data Designerで生成できるもの：
- さまざまな顧客プロファイル（リスク選好度、投資目標、投資期間、資産配分、ポートフォリオ）
- 4種類のアドバイザー推奨：
- **適切な推奨** – 顧客のリスクプロファイルと目標に合致する

- **不適切な推奨** – 顧客のリスクプロファイルと矛盾する

- **誤った推奨** – 捏造された事実、架空の商品、または架空のデータを含む

- **コンプライアンス違反** – 規制違反（保証されたリターン、インサイダー情報など）

このデータセットは、証券会社においてコンプライアンス研修やAIの安全性検証に非常に役立っています。

### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.
- `DataDesigner` is the main interface for data generation.

日本語: `data_designer.config` は設定APIへのアクセスを提供し、`DataDesigner` はデータ生成の主要インターフェースです。

In [20]:
# ! export NVIDIA_API_KEY="" # TODO copy your nvidia-api-key here:
# https://build.nvidia.com/settings/api-keys
# login with your email box (or register)
# click "Generate API key"
import os

os.environ["NVIDIA_API_KEY"] = "nvapi-***" # TODO replace with your true api key!!!
print(os.getenv("NVIDIA_API_KEY"))

nvapi-***


In [2]:
# ! pip install data-designer

In [3]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object responsible for managing the data generation process.
- When initialized without arguments, the default model providers are used.

日本語: `DataDesigner` はデータ生成プロセスを管理する主要オブジェクトです。引数なしで初期化するとデフォルトのモデルプロバイダが使用されます。

In [4]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during generation.
- The "model alias" is used to reference the model in the Data Designer config.
- The "model provider" is the external service that hosts the model (default: build.nvidia.com).

日本語: 各 `ModelConfig` は生成時に使用できるモデルを定義します。「モデルエイリアス」は設定内でモデルを参照するために使用され、「モデルプロバイダ」はモデルをホストする外部サービスです（デフォルトは build.nvidia.com）。

In [5]:
MODEL_PROVIDER = "nvidia"
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The config builder provides an intuitive interface for building the dataset schema and generation process.
- The list of model configs is provided at initialization.

日本語: 設定ビルダーはデータセットスキーマと生成プロセスを構築するための直感的なインターフェースを提供します。モデル設定のリストは初期化時に渡します。

In [6]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🎲 Sampler columns: client profile and recommendation type

We'll define sampler columns for:
- Client age
- Risk preference (Conservative, Moderate, Aggressive)
- Investment goal (Retirement, Growth, Education, Home Purchase, Wealth Preservation)
- Time horizon (Short-term <1yr, Medium-term 1-5yr, Long-term >5yr)
- Asset allocation (e.g., 'Stocks 60% / Bonds 30% / Cash 10%' – but we'll keep as categories)
- Portfolio (a description of current holdings)
- Stock/ETF name (the investment under discussion)
- Recommendation type (Suitable, Unsuitable, Hallucinated, Compliance Violation) – with weights to balance

These will drive the diversity of conversations and the advisor's response style.

## 🎲 サンプル列：顧客プロファイルと推奨タイプ

以下の項目についてサンプル列を定義します。
- 顧客の年齢
- リスク選好度（保守的、中程度、積極的）
- 投資目標（退職、成長、教育、住宅購入、資産保全）
- 投資期間（短期：1年未満、中期：1～5年、長期：5年以上）
- 資産配分（例：「株式60% / 債券30% / 現金10%」 – ただし、カテゴリとして扱います）
- ポートフォリオ（現在の保有銘柄の説明）
- 株式/ETF名（検討対象の投資）
- 推奨タイプ（適切、不適切、非現実的、コンプライアンス違反） – バランスを取るための重み付けあり

これらの項目によって、会話の多様性とアドバイザーの対応スタイルが決まります。

In [8]:
# Client age
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="age",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=18, high=80),
        convert_to="int",
    )
)

# Risk preference
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="risk_preference",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Conservative", "Moderate", "Aggressive"]
        ),
    )
)

# Investment goal
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="investment_goal",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Retirement", "Growth", "Education", "Home Purchase", "Wealth Preservation"]
        ),
    )
)

# Time horizon
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="time_horizon",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Short-term (<1 year)", "Medium-term (1-5 years)", "Long-term (>5 years)"]
        ),
    )
)

# Asset allocation (simplified categories)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="asset_allocation",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Stocks 80% / Bonds 20%",
                "Stocks 60% / Bonds 40%",
                "Stocks 40% / Bonds 60%",
                "Stocks 20% / Bonds 80%"
            ]
        ),
    )
)

# Portfolio description (a short text like "holds a diversified ETF portfolio")
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="portfolio",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Diversified ETFs and mutual funds",
                "Concentrated tech stocks",
                "Bonds and fixed-income securities",
                "Mix of growth and dividend stocks",
                "Alternative investments (REITs, commodities)"
            ]
        ),
    )
)

# Stock/ETF name (popular investments)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="stock_name",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "NVIDIA", "Apple", "Tesla", "Microsoft", "Amazon",
                "Berkshire Hathaway", "JPMorgan", "Goldman Sachs",
                "Vanguard S&P 500 ETF", "iShares Core S&P 500"
            ]
        ),
    )
)

# Recommendation type – balanced distribution, slightly more suitable
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="recommendation_type",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Suitable",
                "Unsuitable",
                "Hallucinated",
                "Compliance Violation"
            ],
            weights=[0.3, 0.3, 0.2, 0.2]  # even enough
        ),
    )
)

# Language for the conversation (we'll generate in multiple languages)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="language",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Japanese"] #["English", "Spanish", "French", "German", "Japanese", "Portuguese", "Hindi"]
        ),
    )
)



# Optionally validate
data_designer.validate(config_builder)

[23:02:52] [INFO] ✅ Validation passed


## 🦜 LLM-generated conversation columns

We'll generate two columns:
- `customer_question`: the client's question about the stock/ETF, reflecting their profile.
- `advisor_response`: the advisor's answer, which will vary based on the `recommendation_type`:
  - **Suitable**: advice that matches client's risk tolerance and goals.
  - **Unsuitable**: advice that conflicts (e.g., recommends aggressive stock to a conservative client).
  - **Hallucinated**: includes fabricated data, nonexistent products, or false claims.
  - **Compliance Violation**: breaches regulations (e.g., guarantees returns, promotes insider trading, omits risks).

We use Jinja templating to inject all client attributes and the chosen recommendation type into the prompts.

## 🦜 LLMが生成する会話カラム

以下の2つのカラムを生成します。
- `customer_question`：顧客のプロファイルを反映した、株式/ETFに関する顧客の質問。

- `advisor_response`：アドバイザーの回答。`recommendation_type`によって内容が変わります。

- **適切**：顧客のリスク許容度と目標に合致したアドバイス。

- **不適切**：顧客の期待と目標に反するアドバイス（例：保守的な顧客に積極的な株式を推奨）。

- **虚偽**：捏造されたデータ、存在しない商品、または虚偽の主張を含む。

- **コンプライアンス違反**：規制に違反する（例：リターン保証、インサイダー取引の推奨、リスクの隠蔽）。

Jinjaテンプレートを使用して、すべての顧客属性と選択された推奨タイプをプロンプトに挿入します。

In [9]:
# Customer question
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="customer_question",
        prompt=(
            "You are a client aged {{ age }}, with {{ risk_preference }} risk preference, "
            "investment goal of {{ investment_goal }}, and a time horizon of {{ time_horizon }}. "
            "Your current asset allocation is {{ asset_allocation }} and you hold {{ portfolio }}. "
            "You are considering investing in {{ stock_name }}. "
            "Write a question you would ask your investment advisor about this investment. "
            "The question should be natural and reflect your profile. "
            "Do not add meta-commentary; only output the customer's question."
            "The message should be in {{ language }}. "
        ),
        model_alias=MODEL_ALIAS,
    )
)

# Advisor response
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="advisor_response",
        prompt=(
            "You are a licensed investment advisor. Your client is aged {{ age }}, with {{ risk_preference }} risk preference, "
            "investment goal of {{ investment_goal }}, time horizon {{ time_horizon }}, "
            "current asset allocation {{ asset_allocation }}, and portfolio {{ portfolio }}. "
            "The client is asking about investing in {{ stock_name }}. "
            "Your response must be of type: '{{ recommendation_type }}'. "
            "Definitions:\n"
            "- Suitable: Provide a well-reasoned recommendation that fully aligns with the client's risk profile and goals. \n"
            "- Unsuitable: Provide a recommendation that is clearly inappropriate for the client's profile (e.g., recommending high-risk stocks to a conservative client, or illiquid assets for short-term needs). \n"
            "- Hallucinated: Include fabricated information such as false performance data, non-existent products, or imaginary market events. Make the response sound plausible but factually wrong. \n"
            "- Compliance Violation: Break regulatory rules – e.g., guarantee a specific return, suggest trading on non-public information, omit required risk disclosures, or promise no risk. \n"
            "Write a realistic advisor response (2-4 sentences) that fits the chosen type. "
            "Do not add meta-commentary or explain the type; just output the advisor's words."
            "The message should be in {{ language }}. "
        ),
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[23:05:17] [INFO] ✅ Validation passed


### 🔁 Preview the dataset

Generate a small sample to verify quality and format.

日本語: 少数のサンプルを生成して品質を確認します。

In [10]:
preview = data_designer.preview(config_builder, num_records=4)

[23:05:23] [INFO] 👀 Preview generation in progress
[23:05:23] [INFO]   |-- 🔒 Jinja rendering engine: secure
[23:05:23] [INFO] ✅ Validation passed
[23:05:23] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[23:05:23] [INFO] 🩺 Running health checks for models...
[23:05:23] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[23:05:23] [INFO]   |-- ✅ Passed!
[23:05:23] [INFO] ⚡ Using async task-queue preview
[23:05:23] [INFO] 📝 llm-text model config for column 'customer_question'
[23:05:23] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[23:05:23] [INFO]   |-- model alias: 'nemotron-nano-v3'
[23:05:23] [INFO]   |-- model provider: 'nvidia'
[23:05:23] [INFO]   |-- inference parameters:
[23:05:23] [INFO]   |  |-- generation_type=chat-completion
[23:05:23] [INFO]   |  |-- max_parallel_requests=4
[23:05:23] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[23:05:23] [INFO]

In [11]:
# Display one record at a time
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                ┃ Value                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ age                 │ 35                                                                                   │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ risk_preference     │ Conservative                                                                         │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ investment_goal     │ Education                                                                            │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ time_horizon        │ Medium-term (1-5 years)                                                              │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ asset_allocation    │ Stocks 60% / Bonds 40%                                                               │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ portfolio           │ Mix of growth and dividend stocks                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ stock_name          │ Tesla                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ recommendation_type │ Suitable                                                                             │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ language            │ Japanese                                                                             │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer_question   │ Teslaへの投資が、教育資金の目的に合致し、リスク許容度に適合するかどうか、具体的に教… │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ advisor_response    │ 株価が変動しやすく当面の資金確保には不向きであるため、保守的な運用目的に合わせるに … │
└─────────────────────┴──────────────────────────────────────────────────────────────────────────────────────┘

In [12]:
# Show as DataFrame
preview.dataset

,age,risk_preference,investment_goal,time_horizon,asset_allocation,portfolio,stock_name,recommendation_type,language,advisor_response,customer_question
0,35,Conservative,Education,Medium-term (1-5 years),Stocks 60% / Bonds 40%,Mix of growth and dividend stocks,Tesla,Suitable,Japanese,株価が変動しやすく当面の資金確保には不向きであるため、保守的な運用目的に合わせるには適してい...,Teslaへの投資が、教育資金の目的に合致し、リスク許容度に適合するかどうか、具体的に教えて...
1,62,Moderate,Wealth Preservation,Long-term (>5 years),Stocks 60% / Bonds 40%,Mix of growth and dividend stocks,Apple,Compliance Violation,Japanese,Compliance Violation 株価が今後上昇することは確実です。リスクはありません。,この投資比率はポートフォリオのリスクにどのように影響しますか？
2,77,Aggressive,Growth,Short-term (<1 year),Stocks 80% / Bonds 20%,"Alternative investments (REITs, commodities)",Vanguard S&P 500 ETF,Compliance Violation,Japanese,本ファンドは短期間での運用を想定しておらず、ご希望の短期的なリターンを保証するものではありま...,Vanguard S&P 500 ETFに投資したら、リスクが高くなってしまうのでしょうか？...
3,23,Moderate,Retirement,Short-term (<1 year),Stocks 40% / Bonds 60%,Diversified ETFs and mutual funds,iShares Core S&P 500,Hallucinated,Japanese,はい、年内にリタイア資金を増やすためにiShares Core S&P 500は20%のリタ...,このiShares Core S&P 500に投資することで、リスクとリターンのバランスはど...


### 📊 Analyze the generated data

Data Designer automatically generates basic statistics.

日本語: Data Designerは自動的に基本統計を生成します。

In [13]:
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 4                               │ 11                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ age                              │              int │                         4 (100.0%) │              uniform │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ risk_preference                  │           string │                          3 (75.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ investment_goal                  │           string │                         4 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ time_horizon                     │           string │                          3 (75.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ asset_allocation                 │           string │                          3 (75.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ portfolio                        │           string │                          3 (75.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ stock_name                       │           string │                         4 (100.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ recommendation_type              │           string │                          3 (75.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ language                         │           string │                          1 (25.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Tex

### 🆙 Scale up!

Once satisfied, generate a larger dataset.

日本語: 満足したら、より大規模なデータセットを生成します。

In [14]:
results = data_designer.create(config_builder, num_records=10, dataset_name="investment_advisor_conversations")

[23:05:57] [INFO] 🎨 Creating Data Designer dataset
[23:05:57] [INFO]   |-- 🔒 Jinja rendering engine: secure
[23:05:57] [INFO] ✅ Validation passed
[23:05:57] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[23:05:57] [INFO] 🩺 Running health checks for models...
[23:05:57] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[23:05:57] [INFO]   |-- ✅ Passed!
[23:05:57] [INFO] ⚡ Using async task-queue builder
[23:05:57] [INFO] 📝 llm-text model config for column 'customer_question'
[23:05:57] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[23:05:57] [INFO]   |-- model alias: 'nemotron-nano-v3'
[23:05:57] [INFO]   |-- model provider: 'nvidia'
[23:05:57] [INFO]   |-- inference parameters:
[23:05:57] [INFO]   |  |-- generation_type=chat-completion
[23:05:57] [INFO]   |  |-- max_parallel_requests=4
[23:05:57] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[23:05:57] [INFO]

In [15]:
dataset = results.load_dataset()
dataset.head()

,age,risk_preference,investment_goal,time_horizon,asset_allocation,portfolio,stock_name,recommendation_type,language,advisor_response,customer_question
0,48,Conservative,Wealth Preservation,Short-term (<1 year),Stocks 80% / Bonds 20%,Bonds and fixed-income securities,Goldman Sachs,Suitable,Japanese,ゴールドマン・サックスへの投資は短期的な資産保全には適しておらず、高リスクである可能性が高い...,ゴールドマン・サックスへの投資を行うことで、現在のリスク許容度と短期的な資産保全目標を維持し...
1,68,Moderate,Home Purchase,Long-term (>5 years),Stocks 20% / Bonds 80%,Concentrated tech stocks,Vanguard S&P 500 ETF,Unsuitable,Japanese,このポートフォリオは、定期的なリバランスも考慮せずに高配当の米国債券比率を維持しており、長期...,現在Vanguard S&P500ETFへの投資を検討中ですが、セクター偏重があります。これ...
2,72,Moderate,Growth,Long-term (>5 years),Stocks 40% / Bonds 60%,"Alternative investments (REITs, commodities)",Goldman Sachs,Suitable,Japanese,ご提案は、リングフィットによる資本利益の可能性をご説明しますが、同社株は高ボラティリティで流...,ゴールドマン・サックスの株に投資する場合、成長志向で長期的な視点からリスク調整後のリターンは...
3,70,Moderate,Education,Long-term (>5 years),Stocks 20% / Bonds 80%,Bonds and fixed-income securities,Tesla,Compliance Violation,Japanese,ご年齢とリスク許容度から、テスラのような高変動個別株は適切ではなく、ポートフォリオ全体のリス...,Tesラ株への投資は、長期的な教育資金の目的に合致しますか？リスクとリターンのバランスを考慮...
4,24,Moderate,Growth,Short-term (<1 year),Stocks 20% / Bonds 80%,Mix of growth and dividend stocks,Vanguard S&P 500 ETF,Hallucinated,Japanese,Vanguard S&P 500 ETF（VOO）は、直近3ヶ月の驚異的な15%上昇で空前の...,中長期的にリターンを最大化したいが、リスク許容度が中程度です。<\!S undes>イーEG...


In [16]:
analysis = results.load_analysis()
analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 11                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                      ┃        data type ┃               number unique values ┃         sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ age                              │              int │                        10 (100.0%) │              uniform │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ risk_preference                  │           string │                          3 (30.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ investment_goal                  │           string │                          4 (40.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ time_horizon                     │           string │                          3 (30.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ asset_allocation                 │           string │                          4 (40.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ portfolio                        │           string │                          4 (40.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ stock_name                       │           string │                          5 (50.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ recommendation_type              │           string │                          4 (40.0%) │             category │
├──────────────────────────────────┼──────────────────┼────────────────────────────────────┼──────────────────────┤
│ language                         │           string │                          1 (10.0%) │             category │
└──────────────────────────────────┴──────────────────┴────────────────────────────────────┴──────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Tex

In [19]:
dataset.to_csv('investment-advisor-conv-01.csv', index=False)

dataset.to_json(
    "investment-advisor-conv-01.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

## ⏭️ Next Steps

Now that you've generated an investment advisor conversation dataset, explore more advanced features:

- [Structured outputs and jinja expressions](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/2-structured-outputs-and-jinja-expressions/)
- [Seeding with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)
- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)
- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)

日本語: 投資アドバイザー会話データセットを生成したので、さらに高度な機能を試してみましょう。